# Sesi?n 2 Apartado 3: Validaci?n cruzada

Objetivo: Entender k-fold CV, `cross_val_score` y evaluar estabilidad de modelos.

## Estrategias y consideraciones

- K-fold CV: divide el dataset en k bloques (folds). Cada fold act?a como conjunto de validaci?n una vez y como parte del conjunto de entrenamiento k-1 veces. La m?trica final es la media y la desviaci?n est?ndar de los scores en los folds.

- StratifiedKFold: variante que preserva la proporci?n de clases en cada fold. Recomendado para clasificaci?n con clases desbalanceadas.

- ?Por qu? usar CV?: estimar la capacidad de generalizaci?n y reducir la varianza de la estimaci?n frente a una sola divisi?n train/test.

- Interpretaci?n de la desviaci?n est?ndar alta entre folds:
  - Indica que el rendimiento es inestable; el modelo puede estar sobreajustando o el dataset puede ser heterog?neo.
  - Investiga con m?s datos, features, regularizaci?n o agrupando por subgrupos relevantes.

- Scoring: elige `scoring` seg?n el objetivo (p.ej. `roc_auc`, `f1`, `precision`). No te quedes solo con `accuracy` si las clases est?n desbalanceadas.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

X, y = make_classification(n_samples=600, n_features=12, random_state=0)

models = {
    "LogisticRegression": LogisticRegression(max_iter=300),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=0),
    "SVM": SVC(),
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

In [ ]:
from sklearn.model_selection import cross_validate

scoring = {"accuracy": "accuracy", "f1": "f1"}

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=scoring,
        return_train_score=False,
    )
    for k in scoring.keys():
        print(f"{name} {k}: mean={scores['test_'+k].mean():.4f}, std={scores['test_'+k].std():.4f}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.metrics import make_scorer, f1_score
f1_scorer = make_scorer(f1_score)

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring=f1_scorer)
    print(f'{name} F1 (stratified 5-fold):', scores.mean(), scores.std())

In [ ]:
# Ejemplos extendidos de validaci?n cruzada
from sklearn.calibration import cross_val_predict
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# cross_val_predict para obtener predicciones 'out-of-fold' y calcular matriz de confusi?n
for name, model in models.items():
    y_oof = cross_val_predict(model, X, y, cv=cv)
    cm = confusion_matrix(y, y_oof)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de confusi?n (out-of-fold) - {name}')
    plt.xlabel('Predicho')
    plt.ylabel('Real')
    plt.show()